In [ ]:
# Task 3 — Event Impact Modeling
Objective: load impact links and events, build an event-indicator association matrix, and run a simple validation test against historical observations (Telebirr / M-Pesa).

In [ ]:
# Imports
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA_RAW = Path('data/raw')
DATA_PROCESSED = Path('data/processed')
OUT_DIR = Path('reports/analysis_outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

In [ ]:
# Load impact links and enriched dataset (events + observations)
impact_fp = DATA_RAW / 'Impact_sheet.csv'
enriched_fp = DATA_PROCESSED / 'ethiopia_fi_enriched_data.csv'
impact = pd.read_csv(impact_fp)
enriched = pd.read_csv(enriched_fp)

# Quick inspect
impact.head()

In [ ]:
# Prepare events table and merge with impact links via parent_id
events = enriched[enriched['record_type'] == 'event'].copy()
events = events.rename(columns={'record_id':'parent_id', 'indicator':'event_name', 'indicator_code':'event_code', 'observation_date':'event_date'})
merged = impact.merge(events[['parent_id','event_name','event_code','event_date']], on='parent_id', how='left')

# Convert numeric magnitude/estimate into a single effect value (prefer impact_magnitude, fallback to impact_estimate)
merged['impact_magnitude_num'] = pd.to_numeric(merged['impact_magnitude'], errors='coerce')
merged['impact_estimate_num'] = pd.to_numeric(merged['impact_estimate'], errors='coerce')
merged['effect_value'] = merged['impact_magnitude_num'].fillna(merged['impact_estimate_num'])

# If direction is 'decrease' store negative values where not already negative
mask_decrease = merged['impact_direction'].str.lower().isin(['decrease','negative'])
merged.loc[mask_decrease & merged['effect_value'].notna() & (merged['effect_value'] > 0), 'effect_value'] *= -1

merged[['record_id','parent_id','event_name','related_indicator','impact_direction','effect_value','lag_months','evidence_basis','comparable_country']].head(20)

In [ ]:
# Build the association matrix: rows=event (parent_id + event_name), cols=related_indicator, values=effect_value
merged['event_label'] = merged['parent_id'] + ' | ' + merged['event_name'].fillna('')
assoc = merged.pivot_table(index='event_label', columns='related_indicator', values='effect_value', aggfunc='first')
# Replace NaN with 0 where no documented effect
assoc_filled = assoc.fillna(0)
assoc_filled.to_csv(OUT_DIR / 'impact_association_matrix.csv')
assoc_filled.head()

In [ ]:
# Visualize the association matrix as a heatmap (absolute magnitude shown)
plt.figure(figsize=(12, max(4, 0.4 * assoc_filled.shape[0])))
sns.heatmap(assoc_filled.replace(0, np.nan), annot=True, fmt='.1f', cmap='RdBu', center=0, linewidths=.5, cbar_kws={'label':'effect (pp or %)'} )
plt.title('Event → Indicator Association Matrix (documented effect values)')
plt.tight_layout()
plt.savefig(OUT_DIR / 'impact_association_heatmap.png', dpi=150)
plt.show()

## Simple impact test: Telebirr / M-Pesa vs observed mobile money account rate
We implement a minimal functional form: documented effect values are interpreted as percentage points (pp) change in the target indicator, applied after the documented lag. If multiple events affect the same indicator, effects sum linearly. This is a strong simplification but suffices for a first-pass check.

In [ ]:
# Helper: get observed values for an indicator by year (from enriched observations)
def observed_by_year(indicator_code):
    df = enriched[enriched['indicator_code'] == indicator_code].copy()
    df = df[df['record_type'] == 'observation']
    # derive year from observation_date when present, else fiscal_year/period_end
    df['observation_date'] = pd.to_datetime(df['observation_date'], errors='coerce')
    df['year'] = df['observation_date'].dt.year.fillna(df['fiscal_year']).astype(int)
    out = df.groupby('year')['value_numeric'].mean().sort_index()
    return out

# Build a minimal time series of yearly observed values for ACC_MM_ACCOUNT and ACC_OWNERSHIP
obs_mm = observed_by_year('ACC_MM_ACCOUNT')
obs_acc = observed_by_year('ACC_OWNERSHIP')
print('Observed Mobile Money Account Rate by year:\n', obs_mm)
print('\nObserved Account Ownership by year:\n', obs_acc)

In [ ]:
# Minimal forward model: apply documented event effects to a baseline year value
def apply_events_simple(indicator_code, base_year, base_value, as_of_year):
    # select links that affect this indicator
    links = merged[merged['related_indicator'] == indicator_code].copy()
    # parse event dates where available
    links['event_date_parsed'] = pd.to_datetime(links['event_date'], errors='coerce')
    # compute event year + lag in years (rounded up)
    links['lag_months'] = pd.to_numeric(links['lag_months'], errors='coerce').fillna(0).astype(float)
    links['effect_year'] = links['event_date_parsed'].dt.year.fillna(np.nan) + (links['lag_months'] / 12.0)
    # assume linear application over 1 year after lag (simplification)
    predicted = base_value
    for _, r in links.iterrows():
        if pd.isna(r['effect_year']):
            continue
        eff_year = int(np.floor(r['effect_year']))
        if eff_year <= as_of_year and r['effect_value']==r['effect_value']:
            # Add effect value (interpreted as percentage points)
            predicted += r['effect_value']
    return predicted, links[['parent_id','event_name','effect_year','effect_value','evidence_basis','comparable_country']]

# Example: test ACC_MM_ACCOUNT between baseline 2021 and 2024 using documented events
base_year = 2021
base_value_mm = float(obs_mm.loc[2021]) if 2021 in obs_mm.index else None
pred_mm_2024, links_mm = apply_events_simple('ACC_MM_ACCOUNT', base_year, base_value_mm, 2024)
print('Baseline (2021) ACC_MM_ACCOUNT =', base_value_mm)
print('Predicted ACC_MM_ACCOUNT by 2024 (simple sum of documented event effects) =', pred_mm_2024)
print('Observed 2024 ACC_MM_ACCOUNT =', float(obs_mm.loc[2024]) if 2024 in obs_mm.index else 'N/A')
print('\nContributing links for ACC_MM_ACCOUNT:\n', links_mm)

# Example: test ACC_OWNERSHIP where Telebirr documented effect exists
base_value_acc = float(obs_acc.loc[2021]) if 2021 in obs_acc.index else None
pred_acc_2024, links_acc = apply_events_simple('ACC_OWNERSHIP', 2021, base_value_acc, 2024)
print('\nBaseline (2021) ACC_OWNERSHIP =', base_value_acc)
print('Predicted ACC_OWNERSHIP by 2024 =', pred_acc_2024)
print('Observed 2024 ACC_OWNERSHIP =', float(obs_acc.loc[2024]) if 2024 in obs_acc.index else 'N/A')
print('\nContributing links for ACC_OWNERSHIP:\n', links_acc)

## Notes / Next steps
- The notebook builds a first-pass association matrix and a very simple additive model that treats documented effect values as immediate percentage-point shifts applied after the documented lag.
- Next steps: (1) incorporate time-distributed impulse responses (e.g., exponential decay or ramp-up), (2) allow multiplicative effects for relative metrics, (3) calibrate uncertain magnitudes using observed time series (least-squares / Bayesian updating).
- I saved the association matrix and heatmap to `reports/analysis_outputs/` as CSV and PNG.